# 05 — ML feature engineering

Create calendar, lag and strictly past-looking rolling features; preserve all Gold rows and persist feature Delta. Target is next-hour pickup_count given known actual history.

**Execution status:** Code is provided for reproducibility. See `results/observed_results.json` for results independently reported from an earlier Fabric run; this generated notebook has not been executed in this environment.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
TABLE="gold_hourly_zone_demand"
gold=spark.table(TABLE)
zone_window=Window.partitionBy("PULocationID").orderBy("pickup_hour")
feat=(gold
 .withColumn("hour_of_day",F.hour("pickup_hour"))
 .withColumn("day_of_week",F.pmod(F.dayofweek("pickup_hour")+5,7))
 .withColumn("day_of_month",F.dayofmonth("pickup_hour"))
 .withColumn("month",F.month("pickup_hour"))
 .withColumn("is_weekend",F.col("day_of_week").isin(5,6).cast("int"))
 .withColumn("lag_1",F.lag("pickup_count",1).over(zone_window))
 .withColumn("lag_2",F.lag("pickup_count",2).over(zone_window))
 .withColumn("lag_24",F.lag("pickup_count",24).over(zone_window))
 .withColumn("lag_168",F.lag("pickup_count",168).over(zone_window))
 .withColumn("rolling_mean_3h",F.round(F.avg("pickup_count").over(zone_window.rowsBetween(-3,-1)),2))
 .withColumn("rolling_mean_24h",F.round(F.avg("pickup_count").over(zone_window.rowsBetween(-24,-1)),2))
)

In [ ]:
print("Features rows:",feat.count())
for c in ["lag_1","lag_2","lag_24","lag_168","rolling_mean_3h","rolling_mean_24h"]:
    print(c,feat.filter(F.col(c).isNull()).count())
assert feat.count()==gold.count()
assert feat.agg(F.sum("pickup_count")).first()[0]==gold.agg(F.sum("pickup_count")).first()[0]

## Persistence

The first 168 hourly observations **per zone** lack `lag_168`; retain them in the saved feature table, then drop them only in downstream modelling. If saving a new version, manually change the table name rather than silently replacing old results.

In [ ]:
FEATURE_TABLE="ml_hourly_demand_features"
if spark.catalog.tableExists(FEATURE_TABLE):
    print("Feature table already exists; not overwritten")
else:
    feat.write.format("delta").mode("errorifexists").saveAsTable(FEATURE_TABLE)
print("Saved feature rows:",spark.table(FEATURE_TABLE).count())